# Basic Features

This workbook demonstrates working with pyetm using the Excel interface for input and output.

Before you begin you will need to place a `config.env` in your root directory. An `../.env.example` is available in the root of the project directory and will guide you through the setup. The `README.md` has additional details on environment management if you're setting up for the first time.

Check the environment is setup correctly.

In [ ]:
# Check the environment is properly configured.
from example_helpers import setup_notebook
import pandas as pd

setup_notebook()

## Understanding the Excel
This section describes the input Excel structure, and how to use it.


| Sheet | Description | Notes |
| --- | --- | --- |
| MAIN | The MAIN section is used to specify inputs for scenarios you load or create. You can provide a short_name to scenarios to make it easier to recognise and interact with them. Otherwise just stick to using the scenario_id to identify your scenario. You can set key metadata like titles and the privacy of scenarios. You can also designate which of the sheets should be referenced for that scenario's custom_curves or sortables - this is designed so you can reference the same sheet for many scenarios, or provide different inputs for different scenarios depending on your use cases. | |
| EXPORT_CONFIG | Pre-determine what will be exported when you call to_excel() on your scenario(s). By default, everything is exported |  |
| USERS | Use the short_name or id for scenarios defined in MAIN to head the columns, and set any user roles, or remove a user, for each scenario as demonstrated in the example excel. |  |
| SLIDER_SETTINGS | Use the short_name or id for scenarios defined in MAIN to head the columns, and set any sliders for each scenario as demonstrated in the example Excel. | |
| GQUERIES | Specify the queries you're interested in requesting for your scenarios. These will apply to all scenarios in case you wish to request results. | |
| CUSTOM_CURVES | Use the given structure to define as many custom curves as you want to apply to each specified scenario. Each custom curve should have 8760 values. | you can name this sheet anything and refer to it in MAIN |
| SORTABLES | Use the given structure to define as many sortables as you want to apply to each specified scenario. | you can name this sheet anything and refer to it in MAIN |


For the purposes of this demonstration workbook, we use the pre-filled template `example_input_excel.xlsx` which demonstrates a few of the possibilities afforded by the pyetm package. If it's your first time using the tool, have a look at the Excel to get a sense of the structure. The input Excel is available in the `/examples/inputs` folder. In pyetm, by default files will be read from the `/inputs` folder and written to the `/outputs` folder.

## Instantiating Scenarios

In the example input you can see the four main ways of instantiating scenarios in the context of pyetm:
1. Load a scenario by ID
2. Create a new scenario
3. Copy a scenario from an existing ID
4. Create a new session

</br>

> <sup><sub> **NOTE:**  Within pyetm, `Scenario` refers to `SavedScenario` and `Session` refers to `Scenario`. Learn more about the difference [in our documentation](https://docs.energytransitionmodel.com/api/saved-scenarios#scenarios-vs-saved-scenarios). We recommend users to use `Scenarios` in pyetm (which thus refers to saved scenarios), and the examples are primarily performed with `Scenarios`. The Scenarios referenced in the example input excel are public scenarios on the pro ETM instance. We recommend changing the scenario_ids in the example to experiment with scenarios you own. </sub></sup>

#### 1. Load a scenario
Load an existing scenario from the ETM. The scenario ID required to load your scenario can be found in the url of the scenario page: https://my.energytransitionmodel.com/saved_scenarios/id.

#### 2. Create a scenario
To create a scenario, leave the id field blank, and make sure you provide an `area_code` and an `end_year`. The scenario will also be created with any other inputs, curves etc you have set in the Excel.

#### 3. Copy a scenario
To copy a scenario, fill a scenario id in the `copy_from` column in the Excel. This will create a new scenario with the same inputs, sortables and curves as the original. The users and preset scenario id will be unset by default.

#### 4. Work with a session
To load, create or copy a `Session` rather than a `Scenario`, set the `session` column to TRUE. In the example we create a `Session`.


## Reading from Excel

In [ ]:
from pyetm.models.scenarios import Scenarios

# No updates mode (default): Load data locally but don't push anything to ETM
scenarios = Scenarios.from_excel("../examples/inputs/example_input_excel.xlsx")

# Update mode: Load and push all data to ETM (creates/updates scenarios)
# scenarios = Scenarios.from_excel("../examples/inputs/example_input_excel.xlsx", update=True)

# Selective update: All options are shown
# scenarios = Scenarios.from_excel("../examples/inputs/example_input_excel.xlsx", update=["user_values", "custom_curves", "sortables", "users"])

**Note:** By default, updates are not applied by calling `from_excel()`. In all modes, data is loaded into local scenario objects. The `update` parameter controls whether API upload calls are made.

## Interacting with Scenarios

Now we have the 'scenarios' in pyetm, which represent actual scenarios in the ETM. One loaded, one created, one copied and one session (created).

The following blocks show how you can explore these scenarios' attributes - run some if you want to explore the data structures.

Show scenario metadata.

In [ ]:
# You can inspect each scenario and print the (meta) info you want
for scenario in scenarios:
    print(f"Title: {scenario.title}")
    print(f"ID: {scenario.id}")
    print(f"Area: {scenario.area_code}")
    print(f"End year: {scenario.end_year}")
    print(f"Version: {scenario.version}")
    print("")

# Or use combine to show them together in a multi-index pandas dataframe
print("Using combine:")
scenarios.combine.main_info()

Display scenario inputs.

In [ ]:
# Inputs are displayed using combine to show them in one multi-index dataframe.
# You may specify the fields you like to see.
# With head(15) we show only the first 15 inputs, but you may use any
# other pandas dataframe methods you prefer.
scenarios.combine.inputs(fields=["user", "default", "min", "max"]).head(15)

Show active couplings (when coupled with an external model).

In [ ]:
# Let's print to see who has a coupling active
print("By looping and checking:")
for scenario in scenarios:
    if not scenario.couplings.empty():
        print(scenario.identifier())
        print(scenario.couplings)
        print("")

# Or we use combine to see all scenarios in a dataframe
print("Or using combine:")
scenarios.combine.couplings()

Show sortables for each scenario.

In [ ]:
# Sortables in a pandas multi index dataframe using combine
scenarios.combine.sortables()

Show custom curves.

In [ ]:
# Custom Curves

# Combine them and show only the first 20 hours in a multi-index dataframe
all_curves_20_hours = scenarios.combine.custom_curves().head(20)

# Show only the interconnector_1_price for all scenarios
all_curves_20_hours.loc[:, (slice(None), "interconnector_1_price")]

Query scenario results.

In [ ]:
# By default combine only shows the future value for a query
print(scenarios.combine.gquery_results())

# If wanted, you can also show the present value. Note that only some queries
# work for the present. By default the present value is 0.
scenarios.combine.gquery_results(columns=["present","future"])

We can directly modify any of the attributes using Pandas, or we can re-export the scenarios to Excel and make modifications that way. When exporting to excel, more data will be available than was in the input, because the ETM results will be included by default. The hourly output curves will be stored in a separate excel workbook, separated by carrier type. By default everything is included, but you can also specify what you want.

Export scenarios to Excel.
This will create scenarios.xlsx and scenarios_hourly_output_curves.xlsx (if you've set hourly_output_curves to true in the output config).

In [ ]:
# Export the scenarios to excel
# scenarios.to_excel("../examples/outputs/scenarios.xlsx") # Export excels takes some time to generate. Uncomment to run.

## Sessions vs. Scenarios

You can also interact with the underlying Session for a Scenario, or save a Session to a Scenario.

In [ ]:
# Get first scenario
scenario = scenarios[0]
session = scenario.session

print(f"Scenario id: {scenario.id}")
print(f"Session id: {session.id}")
print(f"Session id from scenario object: {scenario.session.id}")

# Both will have the same inputs, as both access the session's inputs
print(session.inputs.to_dataframe().head(5))
print(scenario.inputs.to_dataframe().head(5))

 The fourth object in `scenarios` is a Session. By saving it, you will be able to see it in your personal scenarios at: https://my.energytransitionmodel.com/saved_scenarios

In [ ]:
session = scenarios[3]
print(f"Session id: {session.id}")

# Save the session to MyETM as a SavedScenario
scenario = session.save(
      title="My pyetm demo scenario",
      private=True
  )

print(f"Scenario id: {scenario.id}")